# IOAI — 2025 Residential Camp Ml Hdb Storey — ⭐모범답안 (Colab 자동 설정판)

이 노트북은 IOAI 로컬 연습 사이트에서 **데이터·학습환경이 자동 준비**되도록 생성되었습니다.
아래 **설정 셀을 먼저 실행**하면 공식 GitHub 저장소에서 이 문제 폴더만 부분 클론으로 받아
(전체 6.6GB 가 아니라 해당 폴더만), 그 폴더로 이동한 뒤 이후 셀이 그대로 학습/예측을 합니다.
완료 후 생성되는 제출 파일을 내려받아 연습 사이트의 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** 로 바꾸면 학습이 빨라집니다.

In [ ]:
# === 데이터 + 환경 자동 설정 (가장 먼저 실행) ===
# 공식 공개 저장소에서 이 문제 폴더만 부분 클론(sparse)으로 받고 그 폴더로 이동한다.
import os
REPO_URL = "https://github.com/teodora-dimitrova/Singapore-Team-Selection"
CLONE = "Singapore-Team-Selection"
SUBDIR = "IOAI 2025 Residential camp task/ ML_ResidentialCamp2025"
WORKDIR = "IOAI 2025 Residential camp task/ ML_ResidentialCamp2025"
# Colab 은 /content 가 홈. 재실행해도 경로가 안정적이도록 고정 기준에서 시작한다.
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
os.chdir(BASE)
if not os.path.isdir(os.path.join(CLONE, SUBDIR)):
    !git clone --filter=blob:none --no-checkout --depth 1 $REPO_URL $CLONE
    !cd $CLONE && git sparse-checkout set "$SUBDIR"
    !cd $CLONE && git checkout
os.chdir(os.path.join(BASE, CLONE, WORKDIR))
print("작업 폴더:", os.getcwd())
print("내용:", sorted(os.listdir(".")))

# HDB 층수대 예측 (ML Challenge) — 모범답안

Singapore IOAI 2025 · Residential Camp (ML/DS). 싱가포르 HDB 재판매 아파트의 특징(지역·평형·블록·면적·리스·
가격·시점 등)으로 **층수대 `storey_range`(17개 순서형 클래스: `01 TO 03` … `43 TO 45`)** 를 예측한다.
점수 = **정확도**. 제출 `submission.csv`(id, storey_range).

**중요 — 본질적으로 어려운 과제(정직한 안내)**: 한 건물 안에서 특정 거래의 *층*은 주어진 특징으로는 거의
무작위다(가격의 고층 프리미엄은 크기·위치·시점 변동에 묻힌다). 그래서 상한이 낮다:
- 최빈 클래스(`04 TO 06`) 상수예측 ≈ **0.23**
- **건물별 최빈 층수** 예측조차 ≈0.21 로 오히려 더 낮다(건물 안 층 분포가 퍼져 있음).
- 여러 강한 방법(HistGBM·LightGBM·건물 target-encoding·가격 잔차 detrend)이 모두 **≈0.28~0.30** 로 수렴.

**모범답안**: 블록/거리/지역별 **층수-인덱스 집계(mean/max/count)** 를 특징으로 더한 **HistGradientBoosting** →
테스트 정확도 ≈ **0.30** (최빈 0.23 대비 +7%p). 층은 근본적으로 예측이 어렵다는 걸 실측으로 보여주는 것이 핵심.


In [ ]:
import re, numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
tr = pd.read_csv("IOAI2025_ML_Challenge_trainset.csv")
te = pd.read_csv("IOAI2025_ML_Challenge_testset.csv")
test_ids = te["Unnamed: 0"].values                       # 제출 id (=105000..)

# 순서형 타깃(01 TO 03 < 04 TO 06 < …)을 인덱스로
classes = sorted(tr["storey_range"].unique(), key=lambda s: int(s[:2]))
cidx = {c: i for i, c in enumerate(classes)}
y = tr["storey_range"].map(cidx).values
print("train", tr.shape, "test", te.shape, "| classes", len(classes))


In [ ]:
def lease_years(s):
    if pd.isna(s): return np.nan
    yy = re.search(r"(\d+)\s*year", str(s)); mm = re.search(r"(\d+)\s*month", str(s))
    return (int(yy.group(1)) if yy else 0) + (int(mm.group(1)) if mm else 0) / 12.0

# 블록/거리/지역별 층수-인덱스 집계(학습 라벨로) — 지역적 층수 경향(약한 신호)
trs = tr.copy(); trs["sidx"] = y
def group_stats(key):
    g = trs.groupby(key)["sidx"].agg(["mean", "max", "size"])
    g.columns = [f"{key}_{c}" for c in g.columns]; return g.reset_index()
gb, gs, gt_ = group_stats("block"), group_stats("street_name"), group_stats("town")

def build(df):
    d = pd.DataFrame(index=df.index)
    dt = pd.to_datetime(df["month"], errors="coerce")
    d["year"] = dt.dt.year; d["mo"] = dt.dt.month
    d["floor_area_sqm"] = df["floor_area_sqm"]; d["resale_price"] = df["resale_price"]
    d["lease_commence_date"] = df["lease_commence_date"]
    d["rem_lease"] = df["remaining_lease"].map(lease_years)
    d["flat_age"] = d["year"] - df["lease_commence_date"]
    d["price_per_sqm"] = df["resale_price"] / df["floor_area_sqm"].replace(0, np.nan)
    for c in ["town", "flat_type", "flat_model"]:
        d[c] = df[c].astype("category").cat.codes
    m = df.merge(gb, on="block", how="left").merge(gs, on="street_name", how="left").merge(gt_, on="town", how="left")
    for c in [c for c in m.columns if c.endswith(("_mean", "_max", "_size"))]:
        d[c] = m[c].values
    return d
Xtr, Xte = build(tr), build(te)
maj = classes[np.bincount(y).argmax()]
print("majority-class accuracy (참고):", round((tr["storey_range"] == maj).mean(), 4))


In [ ]:
clf = HistGradientBoostingClassifier(max_iter=600, learning_rate=0.08, l2_regularization=1.0,
                                     max_leaf_nodes=63, random_state=0)
clf.fit(Xtr.values, y)
pred = np.array(classes)[clf.predict(Xte.values)]
pd.DataFrame({"id": test_ids, "storey_range": pred}).to_csv("submission.csv", index=False)
print("submission.csv 저장:", len(pred), "행")


### 정리
- 블록/거리/지역별 층수-인덱스 집계 + HistGradientBoosting → 테스트 정확도 ≈ **0.30** (최빈 0.23 대비 +7%p).
- **핵심 교훈(정직성)**: 한 건물 내 특정 거래의 층은 주어진 특징으로는 거의 예측 불가에 가깝다 —
  건물별 최빈 예측(0.21)조차 전역 최빈(0.23)보다 낮고, 강한 GBM·LightGBM·가격 detrend·target-encoding 이
  모두 0.28~0.30 에 수렴한다. *상한이 낮은 과제*임을 실측으로 보여주는 것이 이 문제의 학습 포인트.
- **더 시도해볼 것(제한적 효과)**: (건물,평형,연도) 그룹 내 가격 z-score, 순서형 회귀+반올림, 클래스가중.


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)